# RadioML 2016.10a — Modulation Classification with a CNN (PyTorch)

Classify the modulation scheme of a radio signal from 128 raw I/Q samples.

The dataset ([O'Shea, Corgan & Clancy, 2016](https://pubs.gnuradio.org/index.php/grcon/article/view/11))
holds 11 modulations recorded at 20 SNRs from -20 to +18 dB, 1000 examples each,
every example a 128-sample window cut out of a continuously modulated stream and
passed through a channel model with fading, and with sample-rate and carrier
offsets.

The notebook has two halves:

1. **Sections 3-6 — a classical receiver.** Matched filter, Mueller & Muller
   timing recovery and a Costas loop, applied to QPSK, to show concretely what
   the impairments are and how much work it takes to undo them by hand. This is
   exploratory: on 128-sample windows the loops barely have time to acquire,
   which is the argument for the second half.
2. **Sections 7-12 — a 1D CNN** that skips synchronisation entirely and learns
   from the raw I/Q, evaluated as accuracy against SNR.

The reusable code lives in `src/` (`data.py`, `dsp.py`, `plots.py`); this
notebook is the narrative around it.

## 1. Setup and configuration

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Works after `pip install -e .`; the fallback keeps a plain checkout runnable.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import dsp
from src.data import load_raw, set_seed, split, to_arrays
from src.plots import constellation

set_seed(42)

Every knob the notebook uses is set here, so no downstream cell hides a magic
number and any of them can be re-run after changing one value.

In [ ]:
# --- what to look at -------------------------------------------------------
MOD = "QPSK"        # the recovery in sections 4-6 is QPSK-specific (see src/dsp.py)
SNR_DB = 18         # highest SNR in the dataset: the clearest constellation
NUM_EXAMPLES = 20   # examples overlaid in each plot

# --- receiver front-end ----------------------------------------------------
SPS = dsp.DEFAULT_SPS          # 8 samples per symbol for the digital modulations
ROLLOFF = dsp.DEFAULT_ROLLOFF  # assumed RRC excess bandwidth (0.35)
MM_GAIN = 0.05                 # Mueller & Muller loop gain
LOOP_BW = 0.05                 # Costas natural frequency, rad/symbol
DAMPING = np.sqrt(2) / 2       # critically damped

ALPHA, BETA = dsp.costas_gains(LOOP_BW, DAMPING)
print(f"Costas gains: alpha={ALPHA:.4f}, beta={BETA:.5f}")

## 2. Load the dataset

`load_raw()` downloads the dataset through kagglehub on first use and reads it
from `~/.cache/kagglehub` afterwards. It returns the dict as published:
`(modulation, snr) -> (1000, 2, 128)` array of I/Q samples.

In [ ]:
raw = load_raw()

mods = sorted({mod for mod, _ in raw})
snrs = sorted({snr for _, snr in raw})

print(f"{len(raw)} (modulation, SNR) cells of shape {raw[(MOD, SNR_DB)].shape}")
print(f"{len(mods)} modulations:", mods)
print(f"{len(snrs)} SNRs (dB):", snrs)

## 3. What the raw samples look like

Each example is normalised to unit average power before plotting. The channel
model applies random fading, so amplitudes differ by orders of magnitude between
examples and without normalisation a handful of strong ones would set the scale
for the whole figure.

At 8 samples per symbol only one sample in eight sits at a decision instant; the
other seven are the pulse-shaping filter sweeping between symbols. That is why
the cloud below is a smear rather than four points.

In [ ]:
examples = raw[(MOD, SNR_DB)][:NUM_EXAMPLES]
raw_iq = np.stack([dsp.normalize_power(dsp.to_complex(ex)) for ex in examples])

fig, ax = plt.subplots(figsize=(6, 6))
constellation(
    ax,
    {"Raw samples": raw_iq},
    title=f"{MOD} at {SNR_DB} dB — {NUM_EXAMPLES} examples overlaid",
    alpha=0.05,
)
plt.show()

## 4. Naive downsampling to the symbol rate

The obvious move is to keep one sample in `SPS`. But *which* one? The dataset
gives no timing reference, and the correct sampling phase differs from example
to example.

The grid below takes all 8 possible phases of the same 20 examples. If the
choice were harmless every panel would look alike. It does not: some phases land
near the decision instants and show four lobes, others land mid-transition and
collapse the constellation. Picking a phase by hand is guesswork — which is the
argument for the timing recovery in the next section.

In [ ]:
fig, axes = plt.subplots(2, SPS // 2, figsize=(15, 8), sharex=True, sharey=True)

for phase, ax in enumerate(axes.ravel()):
    constellation(
        ax,
        {f"phase {phase}": raw_iq[:, phase::SPS]},
        title=f"sampling phase {phase}/{SPS}",
        alpha=0.5,
    )
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.suptitle(f"{MOD} at {SNR_DB} dB — every sampling phase gives a different constellation")
fig.tight_layout()
plt.show()

Phase 0 is kept below as the naive baseline to compare the recovered symbols
against. It is an arbitrary choice, not a good one.

In [ ]:
NAIVE_PHASE = 0
naive_symbols = raw_iq[:, NAIVE_PHASE::SPS]

fig, ax = plt.subplots(figsize=(6, 6))
constellation(
    ax,
    {"Raw samples": raw_iq, "Naive downsampling": naive_symbols},
    title=f"{MOD} at {SNR_DB} dB — raw vs. naive downsampling (phase {NAIVE_PHASE})",
    alpha=0.3,
)
plt.show()

## 5. Matched filter and Mueller & Muller timing recovery

Two stages, in the order a receiver applies them.

The **matched filter** is a root-raised-cosine matched to the transmit pulse
shaping. It maximises SNR at the decision instants and, just as importantly, is
what the timing error detector below assumes has already been applied. The
excess bandwidth (0.35) is the GNU Radio default used to generate the dataset:
an assumption, not something measured from the data.

**Mueller & Muller** then estimates the symbol timing itself. It keeps a
fractional sampling instant inside the current symbol period, interpolates the
input there, and moves that instant with a decision-directed error term until
consecutive decisions and samples agree. The timing it produces is continuous,
not snapped to the 1/8-symbol sample grid.

Both run per example: the loops track a single continuous stream, so
concatenating the 20 examples first would ask them to re-acquire at every
boundary. The recovered symbols are concatenated afterwards, only for plotting.

In [ ]:
taps = dsp.root_raised_cosine(SPS, rolloff=ROLLOFF)
filtered_iq = np.stack(
    [dsp.normalize_power(dsp.matched_filter(x, taps)) for x in raw_iq]
)

mm_symbols = np.concatenate(
    [dsp.mueller_muller(x, sps=SPS, gain=MM_GAIN) for x in filtered_iq]
)

print(f"{len(mm_symbols)} symbols recovered from {NUM_EXAMPLES} x {raw_iq.shape[1]} samples")

fig, ax = plt.subplots(figsize=(6, 6))
constellation(
    ax,
    {
        "Raw samples": raw_iq,
        "Naive downsampling": naive_symbols,
        "Mueller & Muller": mm_symbols,
    },
    title=f"{MOD} at {SNR_DB} dB — timing recovery (gain={MM_GAIN})",
    alpha=0.4,
)
plt.show()

## 6. Costas loop — carrier phase and frequency recovery

Correct timing still leaves the constellation rotating: the channel model adds a
carrier frequency offset, so the four lobes smear into a ring. A Costas loop
removes it with a decision-directed phase discriminator driving a second-order
(proportional + integral) filter, which tracks a constant frequency offset with
no steady-state phase error.

Its gains are not tuned by hand. `dsp.costas_gains` derives them from a loop
natural frequency and a damping factor, the two quantities that actually have
meaning — `ALPHA` and `BETA` in section 1 follow from `LOOP_BW = 0.05` and
critical damping.

Two things to keep in mind when reading the result:

* The loop leaves the usual **four-fold phase ambiguity**. Each example settles
  on one of four equally valid quadrant rotations, picked by the noise. The
  recovered constellation is correct up to that rotation, never absolutely
  aligned — harmless for QPSK, which is symmetric under it, but it means these
  symbols carry no absolute phase reference.
* A 128-sample window is about 16 symbols, and both loops need on the order of
  ten symbols to acquire. **What you see is mostly the acquisition transient**,
  not steady-state tracking. Longer records would look considerably cleaner.
  This is the concrete reason the classifier in the second half of the notebook
  is fed raw I/Q instead of recovered symbols.

In [ ]:
mm_costas_symbols = np.concatenate(
    [
        dsp.costas_loop_qpsk(dsp.mueller_muller(x, sps=SPS, gain=MM_GAIN), ALPHA, BETA)
        for x in filtered_iq
    ]
)

fig, ax = plt.subplots(figsize=(6, 6))
constellation(
    ax,
    {
        "Naive downsampling": naive_symbols,
        "Mueller & Muller": mm_symbols,
        "M&M + Costas": mm_costas_symbols,
    },
    title=f"{MOD} at {SNR_DB} dB — carrier recovery (loop bw={LOOP_BW}, damping={DAMPING:.3f})",
    alpha=0.4,
    limit_from="Mueller & Muller",
)
plt.show()

`dsp.recover()` chains all three stages, so the whole front-end on one example
is a single call.

## 7. Build X, y and SNR arrays

## 8. Train / validation / test split

`data.split` stratifies on the `(modulation, SNR)` pair, so every class/SNR cell
is proportionally represented in all three partitions and the accuracy-vs-SNR
curve in section 12 is computed on comparable sample counts.

The default is 60/20/20. The original paper and most follow-up work use a single
50/50 train/test split with no validation set; pass `val_size=0.0,
test_size=0.5` to reproduce that protocol and compare numbers directly.

## 9. PyTorch Dataset and DataLoader

## 10. Model: 1D CNN

## 11. Training loop

## 12. Evaluation: accuracy vs. SNR